In [ ]:
!pip install ego4d


In [ ]:
!ego4d --output_directory "C:\Users\250442661\ego4d_data\v2" --datasets full_scale annotations

In [2]:
import os
import cv2
import json
import torch
import numpy as np
from torch.utils.data import Dataset


class Ego4DDataset(Dataset):

    def __init__(
        self,
        video_dir,
        annotation_file,
        num_frames=16,
        frame_size=224
    ):

        self.video_dir = video_dir
        self.num_frames = num_frames
        self.frame_size = frame_size

        with open(annotation_file, "r") as f:
            self.annotations = json.load(f)

        self.samples = self.build_samples()

    def build_samples(self):

        samples = []

        for item in self.annotations["clips"]:

            video_uid = item["video_uid"]

            clip_uid = item["clip_uid"]

            start_sec = item.get("video_start_sec", 0)
            end_sec = item.get("video_end_sec", 10)

            label = item.get("verb_label", 0)

            video_path = os.path.join(
                self.video_dir,
                f"{video_uid}.mp4"
            )

            if os.path.exists(video_path):

                samples.append({
                    "video_path": video_path,
                    "clip_uid": clip_uid,
                    "start_sec": start_sec,
                    "end_sec": end_sec,
                    "label": label
                })

        return samples

    def sample_frames(
        self,
        video_path,
        start_sec,
        end_sec
    ):

        cap = cv2.VideoCapture(video_path)

        fps = cap.get(cv2.CAP_PROP_FPS)

        start_frame = int(start_sec * fps)
        end_frame = int(end_sec * fps)

        total_frames = end_frame - start_frame

        frame_indices = np.linspace(
            start_frame,
            end_frame - 1,
            self.num_frames,
            dtype=int
        )

        frames = []

        for idx in frame_indices:

            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)

            ret, frame = cap.read()

            if not ret:
                continue

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frame = cv2.resize(
                frame,
                (self.frame_size, self.frame_size)
            )

            frame = frame / 255.0

            frames.append(frame)

        cap.release()

        frames = np.array(frames)

        return frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        frames = self.sample_frames(
            sample["video_path"],
            sample["start_sec"],
            sample["end_sec"]
        )

        frames = torch.tensor(
            frames,
            dtype=torch.float32
        ).permute(0, 3, 1, 2)

        label = torch.tensor(sample["label"])

        return {
            "frames": frames,
            "label": label,
            "clip_uid": sample["clip_uid"]
        }

In [3]:
from torch.utils.data import DataLoader

dataset = Ego4DDataset(
    video_dir="ego4d/videos",
    annotation_file="ego4d/annotations/fho_lta_train.json"
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

batch = next(iter(loader))

print(batch["frames"].shape)
print(batch["label"])

FileNotFoundError: [Errno 2] No such file or directory: 'ego4d/annotations/fho_lta_train.json'

### Extracting visual features

In [1]:
import cv2
import numpy as np


def extract_frames(video_path, sample_rate=8):
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps / sample_rate)

    frames = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_idx % frame_interval == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        frame_idx += 1

    cap.release()

    return np.array(frames)

In [2]:
import os
os.cpu_count()

14

In [ ]:
# import pandas as pd
# from image_processor import process_single_video
# import os
# from concurrent.futures import ProcessPoolExecutor, as_completed
# from tqdm import tqdm

# def process_row(row_data, i, df):
#     """Helper function to process a single row (for parallel execution)"""
#     video_path = row_data['video_path']
#     output_path = os.path.join(
#         r"C:\Users\250442661\ego4d_features\visual_features",
#         f"{os.path.basename(video_path).split('.')[0]}_features_{i}.npy"
#     )
    
#     if i + 1 < len(df) and df.iloc[i + 1]["video_path"] != video_path:
#         next_timestamp = None
#     elif i + 1 < len(df):
#         next_timestamp = df.iloc[i + 1]["timestamp_sec"]
#     else:
#         next_timestamp = None

#     return process_single_video(
#         video_path=video_path,
#         output_path=output_path,
#         timestamp=row_data['timestamp_sec'],
#         next_timestamp=next_timestamp,
#         sample_rate=2
#     )

# df = pd.read_csv(
#     r"C:\Users\250442661\Documents\Ego\ego4d_video_narration_matches.csv"
# )

# # Parallel processing with 8 workers (adjust based on your CPU cores)
# num_workers = 8

# print(f"Starting parallel processing with {num_workers} workers...")
# print(f"Total videos to process: {len(df)}\n")

# with ProcessPoolExecutor(max_workers=num_workers) as executor:
#     # Submit all tasks at once (fast)
#     futures = {
#         executor.submit(process_row, row, i, df): i 
#         for i, row in df.iterrows()
#     }
    
#     # Collect results as they complete with progress bar
#     results = []
#     for future in tqdm(as_completed(futures), total=len(futures), desc="Processing videos", unit="video"):
#         try:
#             result = future.result()
#             results.append(result)
#         except Exception as e:
#             i = futures[future]
#             print(f"✗ Error processing row {i}: {e}")

# print(f"\n✓ Processing complete! Processed {len(results)} videos.")

c:\Users\250442661\Documents\Ego\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 6485.80it/s]


c:\Users\250442661\Documents\Ego\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 6485.80it/s]


Starting parallel processing with 8 workers...
Total videos to process: 16262



In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from image_processor import process_single_video


OUTPUT_DIR = r"C:\Users\250442661\ego4d_features\visual_features"
CSV_PATH = r"C:\Users\250442661\Documents\Ego\ego4d_video_narration_matches.csv"

NUM_WORKERS = 4
SAMPLE_RATE = 2


def process_row(video_path, timestamp, next_timestamp, i):
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    output_path = os.path.join(
        OUTPUT_DIR,
        f"{video_name}_features_{i}.npy"
    )

    if pd.isna(next_timestamp):
        next_timestamp = None

    return process_single_video(
        video_path=video_path,
        output_path=output_path,
        timestamp=timestamp,
        next_timestamp=next_timestamp,
        sample_rate=SAMPLE_RATE
    )


df = pd.read_csv(CSV_PATH)
df = df.head(8000)  # For testing, process only the first 8000 rows
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Precompute next timestamp
df["next_video_path"] = df["video_path"].shift(-1)
df["next_timestamp"] = df["timestamp_sec"].shift(-1)

df.loc[df["next_video_path"] != df["video_path"], "next_timestamp"] = None

tasks = [
    (
        row.video_path,
        row.timestamp_sec,
        row.next_timestamp,
        i
    )
    for i, row in df.iterrows()
]

print(f"Starting processing with {NUM_WORKERS} workers...")
print(f"Total rows to process: {len(tasks)}\n")

results = []
errors = []

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {
        executor.submit(process_row, video_path, timestamp, next_timestamp, i): i
        for video_path, timestamp, next_timestamp, i in tasks
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Processing videos",
        unit="row"
    ):
        i = futures[future]

        try:
            result = future.result()
            results.append(result)
        except Exception as e:
            errors.append((i, str(e)))
            print(f"✗ Error processing row {i}: {e}")

print(f"✓ Processing complete!")
print(f"Successful rows: {len(results)}")
print(f"Failed rows: {len(errors)}")

c:\Users\250442661\Documents\Ego\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 4302.84it/s]


Starting processing with 4 workers...
Total rows to process: 8000


Processing: C:\ego4d\v2\full_scale\77cc4654-4eec-44c6-af05-dbdf71f9a401.mp4

Processing: C:\ego4d\v2\full_scale\77cc4654-4eec-44c6-af05-dbdf71f9a401.mp4

Processing: C:\ego4d\v2\full_scale\77cc4654-4eec-44c6-af05-dbdf71f9a401.mp4

Processing: C:\ego4d\v2\full_scale\77cc4654-4eec-44c6-af05-dbdf71f9a401.mp4


Processing videos:   0%|          | 0/8000 [00:00<?, ?row/s]





























Extracting RGB embeddings: 100%|██████████| 5/5 [00:09<00:00,  1.83s/it]

Extracting RGB embeddings: 100%|██████████| 5/5 [00:09<00:00,  1.81s/it]



Extracting optical flow: 100%|██████████| 4/4 [00:00<00:00, 29.73it/s]

Extracting optical flow: 100%|██████████| 4/4 [00:00<00:00, 29.73it/s]



Ultralytics 8.4.50  Python-3.11.5 torch-2.12.0+cpu CPU (Intel Core Ultra 5 125U)


YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
























Extracting RGB embeddings: 100%|██████████| 10/10 [00:13<00:00,  1.32s/it]






Extracting optical flow: 100%|██████████| 9/9 [00:00<00:00, 32.48it/s]















Processing videos:   0%|          | 1/8000 [00:17<38:25:43, 17.30s/row]

... (3,524 outputs left out)

Saved merged features to: C:\Users\250442661\ego4d_features\visual_features\20bd786f-d222-4b0e-ad2c-a2381929cf78_features_1785.npy
Final feature shape: (5, 779)

Processing: C:\ego4d\v2\full_scale\20bd786f-d222-4b0e-ad2c-a2381929cf78.mp4














Processing videos:  22%|██▏       | 1787/8000 [3:56:32<8:29:17,  4.92s/row]

Saved merged features to: C:\Users\250442661\ego4d_features\visual_features\20bd786f-d222-4b0e-ad2c-a2381929cf78_features_1788.npy
Final feature shape: (3, 779)

Processing: C:\ego4d\v2\full_scale\20bd786f-d222-4b0e-ad2c-a2381929cf78.mp4








Extracting optical flow: 100%|██████████| 2/2 [00:00<00:00, 25.11it/s]




















### Extracting Transcript

In [ ]:
# import whisper
# from pathlib import Path

# # Load model once
# whisper_model = whisper.load_model("base")


# def extract_transcript(video_path):
#     """
#     Generate transcript from video audio using Whisper.
#     """
#     video_path = Path(video_path)

#     if not video_path.exists():
#         raise FileNotFoundError(f"Video not found: {video_path}")

#     result = whisper_model.transcribe(str(video_path))

#     return result["text"]

In [ ]:
# video_path = r"C:\Users\250442661\Documents\Car_review.mp4"

# output_path = r"C:\Users\250442661\ego4d_features\sample_video_transcript.txt"

# transcript = extract_transcript(video_path)

# # Save transcript to file
# # with open(output_path, "w") as f:
# #     f.write(transcript)
    
# print(transcript)

c:\Users\250442661\Documents\Ego\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 So guys, my name is not Forest, but this ugly car behind me is the most tech I have ever seen in a car. So this is the BMW's full size, fully electric i7. It is longer than a Cadillac Escalade, so it's got a lot of room for tech inside, despite looking kind of like a squinting pick. So inside, there are screens powered by wireless Android Auto and CarPlay. There's sensors up here, so you can do this to change the volume. There's also a wireless charger here. There's speakers everywhere, including in the doors. There's a headrest one, and there's woofers in the seats so you can feel the bass. But also, the driving sounds are composed by Hans Zimmer. But what you really want to be is definitely this back seat. So of course, you've got a wireless charger here in front of some hidden cup holders. There's a hidden USB-C port, but there are screens in each of the doors where I can change modes. And if I hit theater mode, it puts up all the shades, including the back and the roof, and drops 

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
import pandas as pd
import os
text_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

CSV_PATH = r"C:\Users\250442661\Documents\Ego\ego4d_video_narration_matches.csv"

OUTPUT_DIR = r"C:\Users\250442661\ego4d_features\textual_features"

df = pd.read_csv(CSV_PATH)
df = df.head(8000)  # For testing, process only the first 8000 rows

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing transcripts"):
    video_path = row["video_path"]
    timestamp = row["timestamp_sec"]

    output_path = os.path.join(
        OUTPUT_DIR,
        f"{os.path.splitext(os.path.basename(video_path))[0]}_transcript_{i}.txt"
    )
    transcript = row["narration_text"]

    embedding = text_model.encode(transcript)

    # Save embedding to file
    np.save(output_path.replace(".txt", "_embedding.npy"), embedding)

Processing transcripts: 100%|██████████| 8000/8000 [03:51<00:00, 34.59it/s]


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
text_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embedding = text_model.encode(transcript)

# Save embedding to file
np.save(output_path.replace(".txt", "_embedding.npy"), embedding)

print(embedding.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6278.81it/s]


(384,)


### Merging Embeddings


In [ ]:
# import numpy as np
# import torch

# from Feature_fusion import CrossAttentionFusion

# VideoEmbedding_path = r"C:\Users\250442661\ego4d_features\sample_video_features_new.npy"
# textEmbedding_path = r"C:\Users\250442661\ego4d_features\sample_video_transcript_embedding.npy"

# visual_np = np.load(VideoEmbedding_path)          # (129, 779)
# text_np = np.load(textEmbedding_path)      # (384,)

# visual_tensor = torch.tensor(
#     visual_np,
#     dtype=torch.float32
# ).unsqueeze(0)   # [1, 129, 779]

# text_tensor = torch.tensor(
#     text_np,
#     dtype=torch.float32
# ).unsqueeze(0)   # [1, 384]

# fusion_model = CrossAttentionFusion(
#     visual_dim=779,
#     text_dim=384,
#     hidden_dim=512,
#     num_heads=8
# )

# fused_features, attention_weights = fusion_model(
#     visual_tensor,
#     text_tensor
# )

# print(fused_features.shape)
# print(attention_weights.shape)

torch.Size([1, 129, 512])
torch.Size([1, 129, 1])


In [3]:
import os
import numpy as np
import torch


# Folder paths
visual_folder = r"C:\Users\250442661\ego4d_features\visual_features"
text_folder = r"C:\Users\250442661\ego4d_features\textual_features"

# Output folder
output_folder = r"C:\Users\250442661\ego4d_features\fused_features"
os.makedirs(output_folder, exist_ok=True)


# Read all visual feature files
video_paths = [
    f for f in os.listdir(visual_folder)
    if f.endswith(".npy")
]

for path in video_paths:
    try:

        # Full paths
        visual_path = os.path.join(visual_folder, path)

        text_file = (
            path.replace("_features_", "_transcript_")
        )

        text_file = text_file.replace(".npy", "_embedding.npy")

        text_path = os.path.join(text_folder, text_file)

        # Load embeddings
        visual_np = np.load(visual_path)      # (129, 779)
        text_np = np.load(text_path)          # (384,)

        # Convert to tensors
        visual_tensor = torch.tensor(
            visual_np,
            dtype=torch.float32
        )

        text_tensor = torch.tensor(
            text_np,
            dtype=torch.float32
        )

        # ---------------------------------------------------
        # Repeat transcript embedding for every frame
        # ---------------------------------------------------

        num_frames = visual_tensor.shape[0]

        # (384,) -> (129, 384)
        repeated_text = text_tensor.unsqueeze(0).repeat(
            num_frames,
            1
        )

        # ---------------------------------------------------
        # Concatenate
        # ---------------------------------------------------

        # (129, 779 + 384)
        fused_features = torch.cat(
            [visual_tensor, repeated_text],
            dim=1
        )

        print("Fused shape:", fused_features.shape)

        # Save fused embedding
        output_path = os.path.join(
            output_folder,
            path.replace("_features_", "_fused_")
        )

        np.save(
            output_path,
            fused_features.numpy()
        )
    except Exception as e:
        print(f"Error processing {path}: {e}")

Fused shape: torch.Size([52, 1163])
Fused shape: torch.Size([1, 1163])
Fused shape: torch.Size([3, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([4, 1163])
Fused shape: torch.Size([3, 1163])
Fused shape: torch.Size([8, 1163])
Fused shape: torch.Size([9, 1163])
Fused shape: torch.Size([4, 1163])
Fused shape: torch.Size([3, 1163])
Fused shape: torch.Size([7, 1163])
Fused shape: torch.Size([7, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([21, 1163])
Fused shape: torch.Size([1, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([5, 1163])
Fused shape: torch.Size([4, 1163])
Fused shape: torch.Size([1, 1163])
Fused shape: torch.Size([4, 1163])
Fused shape: torch.Size([2, 1163])
Fused shape: torch.Size([3, 1163])
Fused shape: torch

In [6]:
fused_np = fused_features.squeeze(0).detach().numpy()

np.save("cross_attention_fused_features.npy", fused_np)

print(fused_np.shape)

(129, 512)
